# HCMUTE Chatbot - Upgrade Pipeline Generation
Chỉnh sửa biến `dataset_path` để chọn dataset nào sẽ được sử dụng cho việc test/generate.
Các pipelines sẽ được chạy trên dataset đó:
1. LLM ONLY 
2. RAG ONLY (Không dùng reranker)
3. OUR RANK (Tích hợp tool Text2SQL, Reranker, và Query Expansion)

In [ ]:
import os
import sys
import asyncio
import pandas as pd

# Thêm đường dẫn thư mục evaluation vào PATH để import config và components
if "evaluation" not in sys.path:
    sys.path.append(os.path.abspath('evaluation'))

from evaluation.config import settings
from evaluation.components.vector_store import get_vector_store
from langchain_qdrant import RetrievalMode

# Import các Pipelines
from evaluation.components.pipelines.llm_only import LLMOnlyPipeline
from evaluation.components.pipelines.basic_rag import BasicRAGPipeline
from evaluation.components.pipelines.our_rag import OurRAGPipeline
from evaluation.components.reranks import JinaReranker

## 1. Lựa chọn Collection & Khởi tạo Vector Store + Reranker

In [ ]:
# Tên collection đã được ingest trước đó bằng file ingest.ipynb
collection_name = "method_human_chunks_hybrid" # thay đổi tại đây nếu bạn ingest tên khác
try:
    vector_store = get_vector_store(mode=RetrievalMode.HYBRID, collection_name=collection_name)
    reranker = JinaReranker()
    print("Khởi tạo Vector Store và Reranker thành công!")
except Exception as e:
    print(f"Lỗi khởi tạo Vector Store hoặc Reranker: {e}")
    print("Hãy kiểm tra lại Qdrant hoặc chạy ingest.ipynb để ingest dataset trước!")

## 2. Khởi tạo các Pipeline cần so sánh

In [ ]:
# 2.1 Pipeline sử dụng LLM không truy xuất dữ liệu
llm_pipeline = LLMOnlyPipeline(model_name="gpt-4o-mini")

# 2.2 Pipeline Basic RAG (Chỉ truy xuất vector cơ bản, không có rerank)
basic_rag_pipeline = BasicRAGPipeline(
    vector_store=vector_store,
    k=10, 
    model_name="gpt-4o-mini",
    reranker=None
)

# 2.3 Pipeline nâng cao (RAG + Qdrant + JinaReranker + Text2SQL + Query Expansion)
our_rag_pipeline = OurRAGPipeline(
    vector_store=vector_store,
    k=10,
    model_name="gpt-4o-mini",
    reranker=reranker,
    rerank_top_k=5,
    use_query_expansion=True
)

print("Khởi tạo 3 Pipelines thành công!")

## 3. Tải dataset cần chạy
Thay đổi biến `dataset_path` phía dưới là đường dẫn CSV tới Dataset của bạn. Output sẽ được xuất tương ứng.

In [ ]:
dataset_path = "evaluation/dataset/evaluation.csv" # Người dùng CÓ THỂ ĐỔI THÀNH BẤT KỲ FILE CSV NÀO
output_path = "evaluation/output/pipeline_results.csv"

df = pd.read_csv(dataset_path)
print(f"Đã tải {len(df)} sample từ {dataset_path}")
df.head()

## 4. Hàm thực thi quá trình sinh (Generation)

In [ ]:
async def run_pipelines_on_dataset(df):
    results = []
    
    for idx, row in df.iterrows():
        print(f"[{idx+1}/{len(df)}] Đang xử lý câu hỏi: {row['question']} ...")
        question = row["question"]
        
        # 1. LLM Only
        res_llm = await llm_pipeline.run(question)
        
        # 2. Basic RAG
        res_basic = await basic_rag_pipeline.run(question)
        
        # 3. Our RAG
        res_our = await our_rag_pipeline.run(question)
        
        results.append({
            "id": row.get("id", idx),
            "question": question,
            "ground_truth_answer": row.get("answer", ""), # Nếu không có ground truth, sẽ lấy chuỗi rỗng
            "generated_llm_only": res_llm.answer,
            "generated_basic_rag": res_basic.answer,
            "generated_our_rag": res_our.answer,
        })
        
    return pd.DataFrame(results)

## 5. Chạy Pipeline và lưu kết quả

In [ ]:
result_df = await run_pipelines_on_dataset(df)
result_df.to_csv(output_path, index=False)
print(f"Hoàn thành! Toàn bộ kết quả sinh câu trả lời đã được lưu vào: {output_path}")

result_df.head()